<a href="https://colab.research.google.com/github/Karthika1022/git-demo/blob/main/13_07_DataModeling_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



#Data Modeling Assessment

  **Points:** 100.  
**Dataset anchor:** E-Commerce Sales Dataset
**ERD tool:** **draw.io / diagrams.net**. Export diagrams as PNG/SVG and submit with the notebook.

This assessment focuses on the first half of the bootcamp: business narrative -> entities -> DDL -> ERD -> normalization -> deliberate denormalization.

**Create a copy of this in your workspace, All the best :) &
Try to not use AI :)**


In [ ]:
# Setup: run first in a fresh Colab runtime
import sqlite3, pandas as pd, re
conn = sqlite3.connect(':memory:')
conn.execute('PRAGMA foreign_keys = ON;')

def sql(q, params=None):
    q = q.strip()
    if q.lower().startswith(('select','pragma','with')):
        return pd.read_sql_query(q, conn, params=params or {})
    conn.executescript(q)
    conn.commit()

def table_exists(name):
    return conn.execute("SELECT 1 FROM sqlite_master WHERE type IN ('table','view') AND name=?", (name,)).fetchone() is not None

def columns(table):
    return [r[1] for r in conn.execute(f"PRAGMA table_info({table})").fetchall()]

def row_count(table):
    return conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

def fk_count(table):
    return len(conn.execute(f"PRAGMA foreign_key_list({table})").fetchall())

def print_check(name, ok, hint):
    print(("PASS" if ok else "FAIL") + " - " + name + ("" if ok else " | Hint: " + hint))

raw_orders_rows = [
    (1001,'2025-01-11','Aija Berzina','aija@example.com','Riga','SKU-201:2, SKU-330:1','Marek Ilves','Baltic North','Credit Card',0.05,0.18),
    (1002,'2025-01-12','Jonas Petras','jonas@example.com','Vilnius','SKU-104:1','Elena Kalnina','Baltic South','Debit Card',0.00,0.22),
    (1003,'2025-01-13','Aija Berzina','aija@example.com','Riga','SKU-201:1, SKU-104:2','Marek Ilves','Baltic North','Wallet',0.10,0.17),
    (1004,'2025-01-14','Maris Ozols','maris@example.com','Liepaja','SKU-330:3','Elena Kalnina','Baltic South','COD',0.00,0.15),
    (1005,'2025-01-15','Greta Jansen','greta@example.com','Tallinn','SKU-510:1, SKU-201:1','Priit Saar','Baltic North','PayPal',0.08,0.20),
    (1006,'2025-01-18','Jonas Petras','jonas@example.com','Vilnius','SKU-330:1, SKU-510:2','Elena Kalnina','Baltic South','Credit Card',0.02,0.21),
    (1007,'2025-02-02','Lina Paulauskaite','lina@example.com','Kaunas','SKU-104:1, SKU-775:1','Elena Kalnina','Baltic South','Debit Card',0.00,0.19),
    (1008,'2025-02-05','Aija Berzina','aija@example.com','Riga','SKU-775:2','Marek Ilves','Baltic North','Wallet',0.04,0.23),
    (1009,'2025-02-07','Greta Jansen','greta@example.com','Tallinn','SKU-201:1','Priit Saar','Baltic North','Credit Card',0.00,0.16),
    (1010,'2025-02-12','Dovile Mockute','dovile@example.com','Klaipeda','SKU-510:1, SKU-330:1','Elena Kalnina','Baltic South','PayPal',0.05,0.20),
    (1011,'2025-03-03','Maris Ozols','maris@example.com','Liepaja','SKU-201:2, SKU-775:1','Elena Kalnina','Baltic South','Credit Card',0.12,0.18),
    (1012,'2025-03-08','Kaisa Lepp','kaisa@example.com','Tartu','SKU-104:3','Priit Saar','Baltic North','Wallet',0.06,0.24),
    (1013,'2025-03-10','Jonas Petras','jonas@example.com','Vilnius','SKU-775:1, SKU-201:1','Elena Kalnina','Baltic South','COD',0.00,0.19),
    (1014,'2025-03-15','Aija Berzina','aija@example.com','Riga','SKU-510:2','Marek Ilves','Baltic North','Credit Card',0.03,0.21),
    (1015,'2025-03-20','Greta Jansen','greta@example.com','Tallinn','SKU-330:2, SKU-104:1','Priit Saar','Baltic North','Debit Card',0.07,0.17),
    (1016,'2025-04-01','Dovile Mockute','dovile@example.com','Klaipeda','SKU-201:1','Elena Kalnina','Baltic South','PayPal',0.00,0.18),
    (1017,'2025-04-05','Kaisa Lepp','kaisa@example.com','Tartu','SKU-510:1, SKU-775:2','Priit Saar','Baltic North','Wallet',0.09,0.22),
    (1018,'2025-04-08','Lina Paulauskaite','lina@example.com','Kaunas','SKU-330:1','Elena Kalnina','Baltic South','Credit Card',0.01,0.16)
]
sql("""
DROP TABLE IF EXISTS raw_orders;
CREATE TABLE raw_orders(order_id INTEGER PRIMARY KEY, order_date TEXT NOT NULL, customer_name TEXT NOT NULL, customer_email TEXT NOT NULL, customer_city TEXT NOT NULL, products_purchased TEXT NOT NULL, sales_rep_name TEXT NOT NULL, sales_rep_region TEXT NOT NULL, payment_method TEXT NOT NULL, discount REAL NOT NULL, profit_margin REAL NOT NULL);
""")
conn.executemany('INSERT INTO raw_orders VALUES (?,?,?,?,?,?,?,?,?,?,?)', raw_orders_rows)
conn.commit()
print('Setup complete. raw_orders rows:', row_count('raw_orders'))
sql('SELECT * FROM raw_orders LIMIT 5')


## Timing

| Section | Minutes | Points |
|---|---:|---:|
| 1. Entities, Attributes & Relationships | 20 | 10 |
| 2. DDL Fundamentals | 20 | 15 |
| 3. Draw.io ER Diagram | 25 | 15 |
| 4. Normalization 1NF -> 3NF | 35 | 20 |
| 5. Denormalization | 15 | 10 |
| 6. New Case Study | 40 | 20 |
| 7. Rapid-fire | 15 | 10 |


## 1. Entities, attributes and relationships — 10 pts, manual

The sales operations team receives e-commerce order extracts. Each row represents an order, but `products_purchased` can contain multiple SKUs in one cell. Customer and sales-rep details repeat across rows. The team needs a clean relational model that supports order processing, product-level analysis, sales-rep reporting and data-quality checks.

**Your task:** list entities, 3-4 attributes per entity, and relationships with cardinality.

### YOUR ANSWER HERE

| Entity | Attributes | Notes |
|---|---|---|
|  |  |  |

| Relationship | Cardinality | Why? |
|---|---|---|
|  |  |  |


## 2. DDL fundamentals — 15 pts, auto-checked

Create `customer`, `product`, and `sales_order`.

Required:
- customer: `customer_id` PK, `customer_email` UNIQUE NOT NULL, `customer_name` NOT NULL, `customer_city` NOT NULL
- product: `sku` PK, `product_name`, `category`, `unit_price` with `CHECK(unit_price >= 0)`
- sales_order: `order_id` PK, `order_date`, `customer_id` FK, `payment_method`, `discount` with check 0..1


In [ ]:
# YOUR ANSWER HERE


In [ ]:
checks=[]
for t in ['customer','product','sales_order']:
    checks.append((f'{t} exists', table_exists(t), f'Create {t}.'))
if table_exists('sales_order'):
    checks.append(('sales_order has FK', fk_count('sales_order') >= 1, 'Add FK from sales_order.customer_id to customer.'))
try:
    conn.executescript('DELETE FROM sales_order; DELETE FROM product; DELETE FROM customer;')
    conn.execute("INSERT INTO customer VALUES (1,'x@example.com','X','Riga')")
    conn.execute("INSERT INTO product VALUES ('SKU-X','Item','Category',10.0)")
    conn.execute("INSERT INTO sales_order VALUES (1,'2025-01-01',1,'Credit Card',0.2)")
    conn.commit(); checks.append(('valid inserts work', True, ''))
except Exception:
    checks.append(('valid inserts work', False, 'Check column order and constraints.'))
try:
    conn.execute("INSERT INTO product VALUES ('SKU-BAD','Bad','Category',-1)"); conn.commit()
    checks.append(('negative unit_price rejected', False, 'Add CHECK(unit_price >= 0).'))
except sqlite3.IntegrityError:
    checks.append(('negative unit_price rejected', True, ''))
try:
    conn.execute("INSERT INTO sales_order VALUES (2,'2025-01-01',999,'Wallet',0.1)"); conn.commit()
    checks.append(('unknown customer rejected', False, 'Foreign key not enforced or missing.'))
except sqlite3.IntegrityError:
    checks.append(('unknown customer rejected', True, ''))
for name, ok, hint in checks: print_check(name, ok, hint)


## 3. Draw.io ER diagram — 15 pts, manual

Build the ERD in **draw.io / diagrams.net** for the schema from Sections 1-2. Include attributes, PKs, FKs, relationship labels and cardinality. Resolve M:N relationships with a junction table.

### YOUR ANSWER HERE

Draw.io export filename:

Assumptions:


## 4. Normalization — 20 pts

### 4a. 1NF — 6 pts, auto-checked
Create `order_line_1nf(order_id, sku, quantity)` by splitting `raw_orders.products_purchased`.


In [ ]:
# YOUR ANSWER HERE


In [ ]:
checks=[('order_line_1nf exists', table_exists('order_line_1nf'), 'Create order_line_1nf.')]
if table_exists('order_line_1nf'):
    checks.append(('required columns exist', set(['order_id','sku','quantity']).issubset(columns('order_line_1nf')), 'Expected order_id, sku, quantity.'))
    checks.append(('split produced line rows', row_count('order_line_1nf') >= 25, 'Split every SKU:quantity pair.'))
    bad=conn.execute('SELECT COUNT(*) FROM order_line_1nf WHERE quantity IS NULL OR quantity <= 0').fetchone()[0]
    checks.append(('quantity positive', bad == 0, 'Quantity should be positive.'))
for name, ok, hint in checks: print_check(name, ok, hint)


### 4b. 2NF — 6 pts, auto-checked
Create `customer_2nf`, `order_header_2nf`, and `order_line_2nf`. Keep order header at one row per order.


In [ ]:
# YOUR ANSWER HERE


In [ ]:
checks=[]
for t in ['customer_2nf','order_header_2nf','order_line_2nf']:
    checks.append((f'{t} exists', table_exists(t), f'Create {t}.'))
if table_exists('customer_2nf'):
    checks.append(('customer emails unique', conn.execute('SELECT COUNT(*)=COUNT(DISTINCT customer_email) FROM customer_2nf').fetchone()[0] == 1, 'One customer per email.'))
if table_exists('order_header_2nf'):
    checks.append(('one order header per raw order', row_count('order_header_2nf') == row_count('raw_orders'), 'Keep order grain.'))
if table_exists('order_line_2nf') and table_exists('order_line_1nf'):
    checks.append(('line count preserved', row_count('order_line_2nf') == row_count('order_line_1nf'), 'Preserve line rows.'))
for name, ok, hint in checks: print_check(name, ok, hint)


### 4c. 3NF — 6 pts, auto-checked
Remove `sales_rep_name -> sales_rep_region` by creating `sales_rep_3nf` and `order_header_3nf` with `sales_rep_id`.


In [ ]:
# YOUR ANSWER HERE


In [ ]:
checks=[]
for t in ['sales_rep_3nf','order_header_3nf']:
    checks.append((f'{t} exists', table_exists(t), f'Create {t}.'))
if table_exists('sales_rep_3nf'):
    checks.append(('sales reps unique', conn.execute('SELECT COUNT(*)=COUNT(DISTINCT sales_rep_name) FROM sales_rep_3nf').fetchone()[0] == 1, 'One row per rep.'))
if table_exists('order_header_3nf'):
    checks.append(('uses sales_rep_id', 'sales_rep_id' in columns('order_header_3nf'), 'Use sales_rep_id in order header.'))
    checks.append(('one 3NF header per raw order', row_count('order_header_3nf') == row_count('raw_orders'), 'Keep order grain.'))
for name, ok, hint in checks: print_check(name, ok, hint)


### 4d. Anomalies — 2 pts, manual

- 1NF anomaly fixed:
- 2NF anomaly fixed:
- 3NF anomaly fixed:


## 5. Denormalization — 10 pts, manual

Ops wants a nightly table with rep name, region, order month, order count, total quantity, average discount and average profit margin.

### YOUR ANSWER HERE

- Reporting table name:
- Grain:
- Fields:
- Trade-offs:
- Pipeline control:


## 6. Case study: university enrollment — 20 pts

A university tracks students, courses, instructors, departments and enrollments. A student can enroll in many courses; a course can have many students. Each enrollment stores term, enrollment date and final grade. Each instructor has one home department, but may teach courses owned by another department. Leaders need average grade per course and instructors teaching outside their home department.

### 6a. Entities and relationships — 5 pts, manual

| Entity | Attributes | Notes |
|---|---|---|
|  |  |  |

| Relationship | Cardinality | Why? |
|---|---|---|
|  |  |  |

### 6b. Draw.io ERD — 5 pts, manual

Draw.io export filename:

### 6c-e. DDL, sample data and SQL — 10 pts, auto-checked
Create `department`, `student`, `instructor`, `course`, `enrollment`; insert sample data; create `v_avg_grade_per_course` and `v_cross_department_instructors`.


In [ ]:
# YOUR ANSWER HERE


In [ ]:
checks=[]
for t in ['department','student','instructor','course','enrollment']:
    checks.append((f'{t} exists', table_exists(t), f'Create {t}.'))
if table_exists('enrollment'):
    checks.append(('enrollment has FKs', fk_count('enrollment') >= 2, 'Enrollment links student and course.'))
    checks.append(('enrollment has grade', 'grade' in columns('enrollment'), 'Grade belongs on enrollment.'))
    checks.append(('sample enrollments', row_count('enrollment') >= 4, 'Insert representative rows.'))
checks.append(('avg grade view exists', table_exists('v_avg_grade_per_course'), 'Create v_avg_grade_per_course.'))
if table_exists('v_avg_grade_per_course'):
    checks.append(('avg grade view returns rows', len(sql('SELECT * FROM v_avg_grade_per_course')) >= 1, 'View should return rows.'))
checks.append(('cross-department view exists', table_exists('v_cross_department_instructors'), 'Create v_cross_department_instructors.'))
if table_exists('v_cross_department_instructors'):
    checks.append(('cross-department view returns rows', len(sql('SELECT * FROM v_cross_department_instructors')) >= 1, 'Insert at least one cross-department scenario.'))
for name, ok, hint in checks: print_check(name, ok, hint)


## 7. Rapid-fire — 10 pts, auto-checked

Set the variables using exact answers: `1NF`, `M:N`, `transitive`, `grain`, `degenerate`, `normalized`, `M:N`, `physical`, `enrollment`, `staleness`.


In [ ]:
q1=''
q2=''
q3=''
q4=''
q5=''
q6=''
q7=''
q8=''
q9=''
q10=''


In [ ]:
answers={'q1':'1NF','q2':'M:N','q3':'transitive','q4':'grain','q5':'degenerate','q6':'normalized','q7':'M:N','q8':'physical','q9':'enrollment','q10':'staleness'}
score=0
for k,v in answers.items():
    ok=globals().get(k)==v; score+=int(ok); print_check(k, ok, 'Use the exact expected answer.')
print(f'Rapid-fire score: {score}/10')


## Manual grading rubric


# Data Modeling Assessment Rubric

| Section | Criterion | Excellent | Good | Adequate | Needs improvement |
|---|---|---|---|---|---|
| 1 | Entities and attributes | Correct entities and attributes; concepts separated cleanly. | Mostly correct; one minor omission. | Core structure present but one modeling error. | Entities largely missing or confused. |
| 1 | Relationships and cardinality | Cardinality explicit and justified; M:N resolved. | Mostly correct with one ambiguity. | Some important cardinality missing/wrong. | Relationship structure unclear. |
| 3 | Draw.io ERD | Clear PK/FK, attributes, relationship labels, cardinality. | Structurally sound with one minor issue. | Diagram readable but one real model issue. | Incomplete or inaccurate. |
| 5 | Denormalization judgement | Grain, fields and trade-offs are clear. | Practical design with minor reasoning gap. | Output exists but grain/risk weak. | Treats denormalization as arbitrary copying. |
| 6a | Case-study model | Complete entities, attributes, cardinality, derived/reporting need. | Mostly correct. | Misses one junction/dependency. | Does not match narrative. |
| 6b | Case-study Draw.io ERD | M:N resolved, keys visible, labels clear. | Clear with minor missing label. | One real relational issue. | Incomplete or inaccurate. |


## 8. Deep capstone case study: Europe Sales Records — OLTP to OLAP end-to-end

**Purpose:** test whether you understand the difference between an operational model and an analytical model, and whether you can connect the full data journey from raw source -> OLTP-style normalized design -> OLAP star schema -> reporting query.

**Dataset:** use KaggleHub to load `mustafabayar/europe-sales-records`. The expected source fields include Region, Country, Item Type, Sales Channel, Order Priority, Order Date, Order ID, Ship Date, Units Sold, Unit Price, Unit Cost, Total Revenue, Total Cost and Total Profit.

**Deliverables:**
1. A short written explanation of the source grain and business process.
2. A normalized OLTP design in draw.io.
3. A dimensional OLAP/star schema design in draw.io.
4. SQL/Pandas implementation that loads prototype normalized and star-schema tables.
5. Two validation queries proving the model supports business reporting.

### 8a. Business understanding and grain — manual

Explain:
- What does one source row represent?
- Which fields are operational identifiers, descriptive attributes and numeric measures?
- Which fields belong naturally in OLTP tables, and which are better suited for OLAP facts/dimensions?

### YOUR ANSWER HERE


In [ ]:
# Deep case study setup: Europe Sales Records via KaggleHub, with a fallback sample.
import os, glob, pandas as pd

try:
    import kagglehub
    path = kagglehub.dataset_download("mustafabayar/europe-sales-records")
    print("Kaggle dataset path:", path)
except Exception as e:
    path = None
    print("KaggleHub download not available in this runtime. Using embedded fallback sample.")
    print("Reason:", type(e).__name__, str(e)[:120])

if path:
    csv_files = glob.glob(os.path.join(path, "*.csv"))
else:
    csv_files = []

if csv_files:
    europe_sales_raw = pd.read_csv(csv_files[0])
else:
    europe_sales_raw = pd.DataFrame([
        ['Europe','Czech Republic','Beverages','Offline','C','9/12/2011',478051030,'9/29/2011',4778,47.45,31.79,226716.10,151892.62,74823.48],
        ['Europe','Bosnia and Herzegovina','Clothes','Online','M','10/14/2013',919133651,'11/4/2013',927,109.28,35.84,101302.56,33223.68,68078.88],
        ['Europe','Austria','Cereal','Offline','C','8/13/2014',987410676,'9/6/2014',5616,205.70,117.11,1155211.20,657689.76,497521.44],
        ['Europe','Bulgaria','Office Supplies','Online','L','10/31/2010',672330081,'11/29/2010',6266,651.21,524.96,4080481.86,3289399.36,791082.50],
        ['Europe','Estonia','Fruits','Online','L','9/28/2016',579463422,'11/1/2016',4958,9.33,6.92,46258.14,34309.36,11948.78],
        ['Europe','France','Cosmetics','Online','H','5/22/2012',314892140,'6/12/2012',1815,437.20,263.33,793518.00,477943.95,315574.05],
        ['Europe','Germany','Household','Offline','M','1/17/2015',221780381,'2/4/2015',3482,668.27,502.54,2327023.14,1749756.28,577266.86],
        ['Europe','Spain','Baby Food','Online','C','7/9/2013',443368995,'8/5/2013',2187,255.28,159.42,558297.36,348651.54,209645.82]
    ], columns=['Region','Country','Item Type','Sales Channel','Order Priority','Order Date','Order ID','Ship Date','Units Sold','Unit Price','Unit Cost','Total Revenue','Total Cost','Total Profit'])

print('Europe sales rows:', len(europe_sales_raw))
display(europe_sales_raw.head())


### 8b. OLTP design — manual + auto-check support

Design a 3NF operational model for order capture. At minimum, include:
- `country_oltp`
- `item_type_oltp`
- `sales_channel_oltp`
- `order_priority_oltp`
- `sales_order_oltp`

Your design should avoid repeating country, item type, sales channel and priority text on every order. Use draw.io for the ERD and paste the export filename below.

### YOUR ANSWER HERE

Draw.io OLTP ERD file:


In [ ]:
# 8b/8c YOUR ANSWER HERE
# Create and populate the OLTP-style normalized tables from europe_sales_raw.


In [ ]:
# Auto-check for 8b/8c. Do not edit.
checks=[]
for t in ['country_oltp','item_type_oltp','sales_channel_oltp','order_priority_oltp','sales_order_oltp']:
    checks.append((f'{t} exists', table_exists(t), f'Create {t}.'))
if table_exists('sales_order_oltp'):
    checks.append(('sales_order_oltp has FKs', fk_count('sales_order_oltp') >= 4, 'Order table should reference country, item type, channel and priority.'))
    checks.append(('order rows loaded', row_count('sales_order_oltp') >= min(5, len(europe_sales_raw)), 'Load rows from europe_sales_raw.'))
for name, ok, hint in checks: print_check(name, ok, hint)


### 8d. OLAP/star schema design — manual + auto-check support

Design a star schema for sales reporting.

Minimum expected tables:
- `dim_date`
- `dim_country`
- `dim_item_type`
- `dim_sales_channel`
- `dim_order_priority`
- `fact_sales`

Declare the fact grain clearly. A good answer treats `Order ID` as a degenerate dimension or source transaction reference and stores numeric measures on the fact.

### YOUR ANSWER HERE

Fact grain:

Draw.io OLAP/star ERD file:


In [ ]:
# 8d/8e YOUR ANSWER HERE
# Create and populate star-schema tables from the OLTP tables or from europe_sales_raw.


In [ ]:
# Auto-check for 8d/8e. Do not edit.
checks=[]
for t in ['dim_date','dim_country','dim_item_type','dim_sales_channel','dim_order_priority','fact_sales']:
    checks.append((f'{t} exists', table_exists(t), f'Create {t}.'))
if table_exists('fact_sales'):
    fact_cols=set(columns('fact_sales'))
    checks.append(('fact has numeric measures', {'units_sold','total_revenue','total_cost','total_profit'}.issubset(fact_cols), 'Fact should include core measures.'))
    checks.append(('fact has dimension keys', len([c for c in fact_cols if c.endswith('_key')]) >= 4, 'Fact should reference dimensions.'))
    checks.append(('fact rows loaded', row_count('fact_sales') >= min(5, len(europe_sales_raw)), 'Load fact rows.'))
for name, ok, hint in checks: print_check(name, ok, hint)


### 8f. End-to-end reporting queries — auto-checked

Create two views:
- `v_profit_by_country_channel`: country, sales_channel, total_revenue, total_profit
- `v_monthly_item_profit`: order_year_month, item_type, total_units, total_profit


In [ ]:
# 8f YOUR ANSWER HERE


In [ ]:
# Auto-check for 8f. Do not edit.
checks=[]
for v in ['v_profit_by_country_channel','v_monthly_item_profit']:
    checks.append((f'{v} exists', table_exists(v), f'Create {v}.'))
    if table_exists(v):
        checks.append((f'{v} returns rows', len(sql(f'SELECT * FROM {v} LIMIT 5')) >= 1, f'{v} should return rows.'))
for name, ok, hint in checks: print_check(name, ok, hint)


### 8g. Final reasoning — manual

In 8-10 lines, explain the difference between your OLTP and OLAP design:
- Why is the OLTP model normalized?
- Why is the OLAP model dimensional?
- What is the declared grain of the fact table?
- Which joins are optimized for operational integrity vs reporting speed?


### YOUR ANSWER HERE


## Additional rubric for Section 8 deep capstone

| Criterion | Excellent | Good | Adequate | Needs improvement |
|---|---|---|---|---|
| OLTP vs OLAP reasoning | Clearly explains operational integrity vs analytical performance, with accurate grain and keys. | Mostly correct, minor missing detail. | Understands basics but confuses one modelling purpose. | Treats OLTP and OLAP as the same shape. |
| Star schema design | Fact grain, dimensions, measures and degenerate order reference are correct. | Strong design with one minor omission. | Fact/dimension split present but weak grain or measures. | No coherent dimensional model. |
| End-to-end implementation | Tables load, views run, and checks reconcile with source. | Tables load with minor naming/quality issues. | Partial implementation. | Does not run or misses core tables. |
